## Exercise 4 - Feature Engineering

In [10]:
import gymnasium as gym
import numpy as np
import time

##### Task 1:

In [2]:
def make_env(max_steps=1000):
    # gym supports max_episode_steps via wrappers
    env = gym.make("MountainCar-v0")
    env = gym.wrappers.TimeLimit(env, max_episode_steps=max_steps)
    return env

Made both Sarsa and Q-learning:

In [3]:
class Discretizer:
    def __init__(self, obs_low, obs_high, n_bins=(30, 30)):
        self.low = obs_low
        self.high = obs_high
        self.n_bins = np.array(n_bins, dtype=int)
        self.bin_width = (self.high - self.low) / self.n_bins

    def encode(self, obs):
        # obs is shape (2,)
        ratios = (obs - self.low) / self.bin_width
        idx = np.floor(ratios).astype(int)
        idx = np.clip(idx, 0, self.n_bins - 1)
        return tuple(idx)  # (pos_bin, vel_bin)

def epsilon_greedy(Q, s_disc, epsilon, n_actions, rng):
    if rng.random() < epsilon:
        return rng.integers(n_actions)
    return int(np.argmax(Q[s_disc]))

def train_q_learning(
    episodes=1000,
    alpha=0.1,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_steps=4000,
    n_bins=(30, 30),
    max_steps=1000,
    seed=0,
):
    rng = np.random.default_rng(seed)
    env = make_env(max_steps=max_steps)
    obs_low = env.observation_space.low
    obs_high = env.observation_space.high
    disc = Discretizer(obs_low, obs_high, n_bins=n_bins)

    n_actions = env.action_space.n
    Q = np.zeros((*n_bins, n_actions), dtype=np.float32)

    # function inside so it gets all initialized variables
    def epsilon_by_episode(ep):
        # linear decay
        frac = min(ep / epsilon_decay_steps, 1.0)
        return epsilon_start + frac * (epsilon_end - epsilon_start)

    returns = []

    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep)
        s = disc.encode(obs)
        eps = epsilon_by_episode(ep)

        total_reward = 0.0
        for t in range(max_steps):
            a = epsilon_greedy(Q, s, eps, n_actions, rng)
            obs2, r, terminated, truncated, _ = env.step(a)
            s2 = disc.encode(obs2)
            total_reward += r

            # Q-learning target: r + gamma * max_a' Q(s',a')
            td_target = r + (0.0 if (terminated or truncated) else gamma * np.max(Q[s2]))
            td_error = td_target - Q[s][a]
            Q[s][a] += alpha * td_error

            s = s2
            if terminated or truncated:
                break

        returns.append(total_reward)

    env.close()
    return Q, returns

def train_sarsa(
    episodes=5000,
    alpha=0.1,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_steps=4000,
    n_bins=(30, 30),
    max_steps=1000,
    seed=0,
):
    rng = np.random.default_rng(seed)
    env = make_env(max_steps=max_steps)
    obs_low = env.observation_space.low
    obs_high = env.observation_space.high
    disc = Discretizer(obs_low, obs_high, n_bins=n_bins)

    n_actions = env.action_space.n
    Q = np.zeros((*n_bins, n_actions), dtype=np.float32)

    def epsilon_by_episode(ep):
        frac = min(ep / epsilon_decay_steps, 1.0)
        return epsilon_start + frac * (epsilon_end - epsilon_start)

    returns = []

    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep)
        s = disc.encode(obs)
        eps = epsilon_by_episode(ep)

        a = epsilon_greedy(Q, s, eps, n_actions, rng)

        total_reward = 0.0
        for t in range(max_steps):
            obs2, r, terminated, truncated, _ = env.step(a)
            s2 = disc.encode(obs2)
            total_reward += r

            if terminated or truncated:
                # terminal update
                td_target = r
                td_error = td_target - Q[s][a]
                Q[s][a] += alpha * td_error
                break

            a2 = epsilon_greedy(Q, s2, eps, n_actions, rng)

            # SARSA target: r + gamma * Q(s', a')
            td_target = r + gamma * Q[s2][a2]
            td_error = td_target - Q[s][a]
            Q[s][a] += alpha * td_error

            s, a = s2, a2

        returns.append(total_reward)

    env.close()
    return Q, returns

def evaluate(Q, n_bins=(30, 30), episodes=20, max_steps=1000, seed=123):
    env = make_env(max_steps=max_steps)
    disc = Discretizer(env.observation_space.low, env.observation_space.high, n_bins=n_bins)
    returns = []

    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep)
        s = disc.encode(obs)
        total_reward = 0.0

        for t in range(max_steps):
            a = int(np.argmax(Q[s]))
            obs2, r, terminated, truncated, _ = env.step(a)
            s = disc.encode(obs2)
            total_reward += r
            if terminated or truncated:
                break

        returns.append(total_reward)

    env.close()
    return float(np.mean(returns)), float(np.std(returns))



In [4]:
# Q-learning
Q, train_returns = train_q_learning(
    episodes=8000,
    alpha=0.15,
    gamma=0.99,
    n_bins=(30, 30),
    max_steps=1000,
    seed=0,
)
mean_ret, std_ret = evaluate(Q, n_bins=(30, 30), episodes=50, max_steps=1000)
print("Q-learning eval return:", mean_ret, "+/-", std_ret)

Q-learning eval return: -191.02 +/- 22.274191343346228


In [5]:
# SARSA 
Qs, sarsa_returns = train_sarsa(
    episodes=8000,
    alpha=0.15,
    gamma=0.99,
    n_bins=(30, 30),
    max_steps=1000,
    seed=0,
)
mean_ret, std_ret = evaluate(Qs, n_bins=(30, 30), episodes=50, max_steps=1000)
print("SARSA eval return:", mean_ret, "+/-", std_ret)

SARSA eval return: -168.28 +/- 19.36702351937437


In [12]:
def watch_policy(Q, n_bins=(30,30), max_steps=1000, seed=0):
    env = gym.make("MountainCar-v0", render_mode="human")
    env = gym.wrappers.TimeLimit(env, max_episode_steps=max_steps)

    obs_low = env.observation_space.low
    obs_high = env.observation_space.high
    bin_width = (obs_high - obs_low) / np.array(n_bins, dtype=int)

    def discretize(obs):
        idx = np.floor((obs - obs_low) / bin_width).astype(int)
        idx = np.clip(idx, 0, np.array(n_bins) - 1)
        return tuple(idx)

    obs, _ = env.reset(seed=seed)
    s = discretize(obs)

    for _ in range(max_steps):
        a = int(np.argmax(Q[s]))
        obs, r, terminated, truncated, _ = env.step(a)
        s = discretize(obs)
        if terminated or truncated:
            break

    env.close()

# Use your trained Qs
watch_policy(Qs, n_bins=(30,30), max_steps=1000, seed=1)

In [ ]:
def watch_policy(Q, n_bins=(30,30), max_steps=1000, seed=1, step_delay=0.01):
    env = gym.make("MountainCar-v0", render_mode="human")
    env = gym.wrappers.TimeLimit(env, max_episode_steps=max_steps)

    obs_low = env.observation_space.low
    obs_high = env.observation_space.high
    bin_width = (obs_high - obs_low) / np.array(n_bins, dtype=int)

    def discretize(obs):
        idx = np.floor((obs - obs_low) / bin_width).astype(int)
        idx = np.clip(idx, 0, np.array(n_bins) - 1)
        return tuple(idx)

    obs, _ = env.reset(seed=seed)
    s = discretize(obs)

    # draw first frame
    env.render()
    time.sleep(0.2)

    for _ in range(max_steps):
        a = int(np.argmax(Q[s]))
        obs, r, terminated, truncated, _ = env.step(a)
        s = discretize(obs)

        # force drawing + keep window responsive
        env.render()
        time.sleep(step_delay)

        if terminated or truncated:
            break

    env.close()

watch_policy(Qs, n_bins=(30,30), max_steps=1000, seed=1)

##### Task 2:

In [6]:
bin_settings = [
    (10, 10),
    (20, 20),
    (30, 30),
    (40, 40),
    (50, 50),
]

In [7]:
results = []
for n_bins in bin_settings:
    Q, train_returns = train_sarsa(
        episodes=8000,
        alpha=0.15,
        gamma=0.99,
        n_bins=n_bins,
        max_steps=1000,
        seed=0,
    )
    mean_ret, std_ret = evaluate(Q, n_bins=n_bins, episodes=50, max_steps=1000, seed=123)
    
    # compute the implied deltas (optional but nice for your report)
    delta_pos = 1.8 / n_bins[0]
    delta_vel = 0.14 / n_bins[1]

    results.append((n_bins, delta_pos, delta_vel, mean_ret, std_ret))

for row in results:
    print(
        f"n_bins={row[0]}  Δpos={row[1]:.4f}  Δvel={row[2]:.6f}  "
        f"eval_return={row[3]:.2f} +/- {row[4]:.2f}"
    )

n_bins=(10, 10)  Δpos=0.1800  Δvel=0.014000  eval_return=-200.00 +/- 0.00
n_bins=(20, 20)  Δpos=0.0900  Δvel=0.007000  eval_return=-182.70 +/- 25.25
n_bins=(30, 30)  Δpos=0.0600  Δvel=0.004667  eval_return=-168.28 +/- 19.37
n_bins=(40, 40)  Δpos=0.0450  Δvel=0.003500  eval_return=-149.66 +/- 30.05
n_bins=(50, 50)  Δpos=0.0360  Δvel=0.002800  eval_return=-187.94 +/- 18.79


##### Task 3:
Yes, we could try normalizing and linear function approximation. Just scale position and velocity to [-1,1] (or mean 0, std 1), then learn with a linear approximator. It might be simple, but it helps stability and makes step sizes easier to tune.

Another possible improvement is to use multiple offset grids (tilings) instead of a single discretization grid. The state is then represented by the set of active tiles across these tilings. This approach provides smoother generalization than simple binning while still allowing the use of linear or tabular-style updates, and it is commonly applied in problems such as MountainCar.